# Experiment 7: Scale-up on public benchmarks (real human queries, 4 datasets x 3 embedding models)

Addresses the main reviewer concerns: single domain, tiny scale, synthetic queries, single embedding model.

- **Datasets (BEIR, public, real queries):** SciFact, NFCorpus, ArguAna, FiQA-2018 (~5k to 57k docs, up to 1000 queries each)
- **Embedding models:** MiniLM-L6, mpnet-base, bge-small (reranker fixed: ms-marco MiniLM cross-encoder)
- **Strategies:** Dense, Hybrid (alpha=0.75, fixed), Hybrid+Rerank, Oracle, Rule router, **Learned cost-aware router**, score-margin (confidence-threshold) baselines
- **Metrics:** nDCG@10 (primary), Recall@10, MRR@10, measured latency (ms/query, includes router overhead)
- **Extras:** paired bootstrap CIs, Pareto (quality vs latency) curves, **leave-one-dataset-out router transfer**, LaTeX tables

**Use a GPU runtime** (Runtime > Change runtime type > T4). Outputs are saved straight to Google Drive (`AI_Project/RAG Research/Exp7`). Estimated run time is roughly 30-60 min on a T4.

In [1]:
!pip install -q sentence-transformers beir scikit-learn scipy pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.9/304.9 kB 22.7 MB/s eta 0:00:00


## Shared helpers (BM25, metrics, features, routing analysis, reporting)

In [2]:
import numpy as np, pandas as pd, re, json, time, os
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import KFold
from sklearn.ensemble import GradientBoostingRegressor

METHODS = ['dense', 'hybrid', 'rerank']          # ordered cheapest -> most expensive
LAMBDAS = np.r_[0.0, np.geomspace(0.005, 1.0, 14)]

# ---------------------------------------------------------------- BM25 (fast, sparse, vectorised)
class BM25Index:
    def __init__(self, docs, k1=1.2, b=0.75):
        self.cv = CountVectorizer(stop_words='english', lowercase=True)
        tf = self.cv.fit_transform(docs).tocsr().astype(np.float32)
        N = tf.shape[0]
        df = np.asarray((tf > 0).sum(axis=0)).ravel()
        self.idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
        dl = np.asarray(tf.sum(axis=1)).ravel()
        avgdl = max(dl.mean(), 1e-9)
        tf = tf.tocoo()
        denom = tf.data + k1 * (1 - b + b * dl[tf.row] / avgdl)
        data = tf.data * (k1 + 1) / denom * self.idf[tf.col]
        self.M = sparse.csr_matrix((data, (tf.row, tf.col)), shape=tf.shape)
        self.MT = self.M.T.tocsr()

    def score(self, queries, batch=256):
        out = []
        for s in range(0, len(queries), batch):
            Q = self.cv.transform(queries[s:s + batch]).astype(np.float32).tocsr()
            Q.data[:] = 1.0
            out.append((Q @ self.MT).toarray().astype(np.float32))
        return np.vstack(out)

    def query_stats(self, qs):
        an = self.cv.build_analyzer(); vocab = self.cv.vocabulary_
        rows = []
        for q in qs:
            toks = an(q)
            inv = [t for t in toks if t in vocab]
            idfs = [float(self.idf[vocab[t]]) for t in inv]
            rows.append(dict(
                n_tok=len(q.split()), n_terms=len(toks),
                frac_oov=1 - len(inv) / max(1, len(toks)),
                mean_idf=float(np.mean(idfs)) if idfs else 0.0,
                max_idf=max(idfs) if idfs else 0.0,
                has_digit=float(bool(re.search(r'\d', q))),
                has_special=float(bool(re.search(r'[_/\\.\-:=]', q))),
                n_chars=len(q)))
        return pd.DataFrame(rows)

# ---------------------------------------------------------------- array utils
def minmax_rows(S):
    mn = S.min(1, keepdims=True); mx = S.max(1, keepdims=True)
    return (S - mn) / (mx - mn + 1e-9)

def topk(S, k):
    k = min(k, S.shape[1])
    part = np.argpartition(-S, k - 1, axis=1)[:, :k]
    vals = np.take_along_axis(S, part, 1)
    order = np.argsort(-vals, axis=1)
    return np.take_along_axis(part, order, 1)

def topvals(S, k=20):
    k = min(k, S.shape[1])
    return -np.sort(-np.partition(S, S.shape[1] - k, axis=1)[:, -k:], axis=1)

# ---------------------------------------------------------------- metrics (graded relevance supported)
def ndcg_at_k(ranked, rel, k=10):
    dcg = sum(rel.get(i, 0) / np.log2(r + 2) for r, i in enumerate(ranked[:k]))
    ideal = sorted(rel.values(), reverse=True)[:k]
    idcg = sum(g / np.log2(r + 2) for r, g in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

def eval_lists(tops, rels, k=10):
    nd, rc, mr = [], [], []
    for lst, rel in zip(tops, rels):
        lst = list(lst[:k])
        nd.append(ndcg_at_k(lst, rel, k))
        pos = {i for i, g in rel.items() if g > 0}
        rc.append(len(pos & set(lst)) / max(1, len(pos)))
        r_ = 0.0
        for r, i in enumerate(lst):
            if i in pos:
                r_ = 1.0 / (r + 1); break
        mr.append(r_)
    return np.array(nd), np.array(rc), np.array(mr)

# ---------------------------------------------------------------- retrieval stack (Dense / Hybrid / Hybrid+Rerank)
def run_stack(model_name, doc_texts, q_texts, bm, reranker, exclude_idx=None, alpha=0.75,
              shortlist=20, q_prefix='', device=None, out_k=10):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name, device=device)
    model.max_seq_length = 256
    D = model.encode(doc_texts, batch_size=128, normalize_embeddings=True,
                     convert_to_numpy=True, show_progress_bar=True)
    Q = model.encode([q_prefix + q for q in q_texts], batch_size=128, normalize_embeddings=True,
                     convert_to_numpy=True, show_progress_bar=False)
    S_d = (Q @ D.T).astype(np.float32)
    S_b = bm.score(q_texts)
    if exclude_idx is not None:                       # ignore identical ids (e.g. ArguAna)
        for i, j in enumerate(exclude_idx):
            if j is not None:
                S_d[i, j] = S_d[i].min(); S_b[i, j] = S_b[i].min()
    H = alpha * minmax_rows(S_d) + (1 - alpha) * minmax_rows(S_b)
    top_d = topk(S_d, out_k)
    top_h_full = topk(H, shortlist)
    pairs = [(q_texts[i], doc_texts[j]) for i in range(len(q_texts)) for j in top_h_full[i]]
    sc = np.asarray(reranker.predict(pairs, batch_size=64, show_progress_bar=True)).reshape(len(q_texts), -1)
    order = np.argsort(-sc, axis=1)
    top_r = np.take_along_axis(top_h_full, order, 1)[:, :out_k]
    return dict(S_d=S_d, S_b=S_b, H=H, top_d=top_d, top_h=top_h_full[:, :out_k], top_r=top_r,
                model=model, D=D)

def measure_latency(model, D, bm, reranker, doc_texts, q_sample, alpha=0.75, shortlist=20, q_prefix=''):
    """Per-query wall-clock latency in ms for [dense, hybrid, rerank] (each includes the previous stage)."""
    for q in q_sample[:3]:
        model.encode([q_prefix + q], normalize_embeddings=True, show_progress_bar=False)
    td, th, tr = [], [], []
    for q in q_sample:
        t0 = time.perf_counter()
        qe = model.encode([q_prefix + q], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
        sd = (qe @ D.T).astype(np.float32)
        _ = topk(sd, 10)
        t1 = time.perf_counter()
        sb = bm.score([q])
        h = alpha * minmax_rows(sd) + (1 - alpha) * minmax_rows(sb)
        sh = topk(h, shortlist)[0]
        t2 = time.perf_counter()
        _ = reranker.predict([(q, doc_texts[j]) for j in sh], show_progress_bar=False)
        t3 = time.perf_counter()
        td.append(t1 - t0); th.append(t2 - t0); tr.append(t3 - t0)
    return [1000 * float(np.mean(td)), 1000 * float(np.mean(th)), 1000 * float(np.mean(tr))]

# ---------------------------------------------------------------- label-free routing features
def build_features(S_d, S_b, H, q_texts, bm):
    d, b, h = topvals(S_d), topvals(S_b), topvals(H)
    kk = d.shape[1]
    f = pd.DataFrame({
        'd_top1': d[:, 0], 'd_margin': d[:, 0] - d[:, 1], 'd_spread': d[:, 0] - d[:, kk - 1], 'd_std': d.std(1),
        'b_top1': b[:, 0], 'b_margin': (b[:, 0] - b[:, 1]) / (b[:, 0] + 1e-6),
        'b_spread': (b[:, 0] - b[:, kk - 1]) / (b[:, 0] + 1e-6),
        'h_margin': h[:, 0] - h[:, 1], 'h_spread': h[:, 0] - h[:, kk - 1]})
    td, tb = topk(S_d, 10), topk(S_b, 10)
    f['overlap10'] = [len(set(a) & set(c)) / 10.0 for a, c in zip(td, tb)]
    f['top1_agree'] = (td[:, 0] == tb[:, 0]).astype(float)
    qs = bm.query_stats(q_texts)
    return pd.concat([qs.reset_index(drop=True), f.reset_index(drop=True)], axis=1)

# ---------------------------------------------------------------- routing analysis
def router_cost(choice, t):
    t = np.asarray(t, float)
    base = np.full(len(choice), t[1])                 # features need dense + BM25 scores
    base[choice == 2] = t[2]
    return float(base.mean())

def bootstrap_diff(a, b, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed); nq = len(a)
    idx = rng.integers(0, nq, size=(n_boot, nq))
    d = a[idx].mean(1) - b[idx].mean(1)
    lo, hi = np.percentile(d, [2.5, 97.5])
    return dict(diff=float(a.mean() - b.mean()), ci_lo=float(lo), ci_hi=float(hi), p_le0=float((d <= 0).mean()))

def fit_gain_models(Xtr, Ytr, seed=42):
    models = {}
    for m in (1, 2):
        reg = GradientBoostingRegressor(n_estimators=150, max_depth=2, learning_rate=0.05,
                                        subsample=0.8, random_state=seed)
        reg.fit(Xtr, Ytr[:, m] - Ytr[:, 0]); models[m] = reg
    return models

def predict_gains(models, Xte):
    P = np.zeros((len(Xte), 3))
    for m, reg in models.items():
        P[:, m] = reg.predict(Xte)
    return P

def choose(P, lam, t):
    cn = np.asarray(t, float) / t[2]
    U = P - lam * (cn - cn[0])[None, :]
    U[:, 0] = 0.0
    return U.argmax(1)

def _row(nd, rc, mr, cost):
    return dict(ndcg=float(nd.mean()), recall=float(rc.mean()), mrr=float(mr.mean()), cost=float(cost))

def analyze_routing(ND, RC, MR, t, X, seed=42, n_boot=5000, n_splits=5):
    nq = len(ND); ar = np.arange(nq); t = np.asarray(t, float)
    Xv = X.values.astype(float)
    res = {'n_queries': nq, 't': t.tolist()}
    best_fixed = int(ND.mean(0).argmax())
    tab = {}
    for i, name in enumerate(['Dense', 'Hybrid', 'Rerank']):
        tab[name] = _row(ND[:, i], RC[:, i], MR[:, i], t[i])
    # rule router (label-free, query text only)
    ntok = X['n_tok'].values; has_d = X['has_digit'].values > 0
    rc_ = np.where(has_d & (ntok < 12), 0, np.where(ntok >= 14, 1, 2))
    tab['Rule router'] = _row(ND[ar, rc_], RC[ar, rc_], MR[ar, rc_], t[rc_].mean())
    # oracle (ties -> cheapest because columns are ordered by cost)
    oc = ND.argmax(1)
    tab['Oracle'] = _row(ND[ar, oc], RC[ar, oc], MR[ar, oc], t[oc].mean())
    res['oracle_label_counts'] = {METHODS[i]: int((oc == i).sum()) for i in range(3)}
    res['n_informative'] = int((ND.max(1) - ND.min(1) > 1e-9).sum())
    # learned router: out-of-fold predicted gain over dense
    P = np.zeros((nq, 3))
    kf = KFold(n_splits=min(n_splits, nq), shuffle=True, random_state=seed)
    for tr, te in kf.split(Xv):
        P[te] = predict_gains(fit_gain_models(Xv[tr], ND[tr], seed), Xv[te])
    sweep = []
    for lam in LAMBDAS:
        ch = choose(P, lam, t)
        r = _row(ND[ar, ch], RC[ar, ch], MR[ar, ch], router_cost(ch, t))
        r.update(lam=float(lam), frac_rerank=float((ch == 2).mean()), frac_dense=float((ch == 0).mean()))
        sweep.append(r)
    ch0 = choose(P, 0.0, t)
    tab['Learned router'] = _row(ND[ar, ch0], RC[ar, ch0], MR[ar, ch0], router_cost(ch0, t))
    res['learned_sweep'] = sweep
    res['best_fixed'] = ['Dense', 'Hybrid', 'Rerank'][best_fixed]
    res['boot_learned_vs_best_fixed'] = bootstrap_diff(ND[ar, ch0], ND[:, best_fixed], n_boot, seed)
    bf_nd = ND[:, best_fixed].mean()
    ok = [s['cost'] for s in sweep if s['ndcg'] >= bf_nd]
    res['min_cost_to_match_best_fixed'] = float(min(ok)) if ok else None
    res['oracle_gap_closed'] = float((tab['Learned router']['ndcg'] - bf_nd) /
                                     max(tab['Oracle']['ndcg'] - bf_nd, 1e-9))
    # confidence-threshold (score-margin) baselines, swept as Pareto curves
    conf = X['d_margin'].values; order = np.argsort(conf)
    curves = {}
    for name, base in [('Margin router (dense/rerank)', 0), ('Margin router (hybrid/rerank)', 1)]:
        pts = []
        for q in np.linspace(0, 1, 21):
            k = int(round(q * nq)); ch = np.full(nq, base); ch[order[:k]] = 2
            pts.append(dict(q=float(q), ndcg=float(ND[ar, ch].mean()), cost=float(t[ch].mean())))
        curves[name] = pts
    res['margin_curves'] = curves
    res['table'] = tab
    res['_P'] = P
    return res

def lodo_routing(sets, n_boot=5000, seed=42):
    """Leave-one-set-out: train the gain regressors on all other sets, test on the held-out one."""
    out = {}
    names = list(sets.keys())
    if len(names) < 2:
        return out
    for held in names:
        tr_names = [k for k in names if k != held]
        Xtr = np.vstack([sets[k]['X'].values.astype(float) for k in tr_names])
        Ytr = np.vstack([sets[k]['ND'] for k in tr_names])
        models = fit_gain_models(Xtr, Ytr, seed)
        S = sets[held]; ND = S['ND']; ar = np.arange(len(ND)); t = np.asarray(S['t'], float)
        P = predict_gains(models, S['X'].values.astype(float))
        sweep = []
        for lam in LAMBDAS:
            ch = choose(P, lam, t)
            sweep.append(dict(lam=float(lam), ndcg=float(ND[ar, ch].mean()), cost=router_cost(ch, t)))
        bf = int(ND.mean(0).argmax()); ch0 = choose(P, 0.0, t)
        out[held] = dict(best_fixed=['Dense', 'Hybrid', 'Rerank'][bf], best_fixed_ndcg=float(ND[:, bf].mean()),
                         router_ndcg=float(ND[ar, ch0].mean()), sweep=sweep,
                         boot=bootstrap_diff(ND[ar, ch0], ND[:, bf], n_boot, seed))
    return out

# ---------------------------------------------------------------- reporting (tables, LaTeX, figures)
ROW_ORDER = ['Dense', 'Hybrid', 'Rerank', 'Rule router', 'Learned router', 'Oracle']

def main_table(RES_model, metric='ndcg'):
    dsets = list(RES_model.keys())
    df = pd.DataFrame({d: {r: RES_model[d]['table'][r][metric] for r in ROW_ORDER} for d in dsets})
    df['Avg'] = df.mean(axis=1)
    if metric != 'cost':
        df['Latency (ms)'] = pd.Series({r: np.mean([RES_model[d]['table'][r]['cost'] for d in dsets]) for r in ROW_ORDER})
    return df.loc[ROW_ORDER]

def _esc(s):
    return str(s).replace('_', '\\_').replace('%', '\\%').replace('&', '\\&')

def df_to_latex(df, caption, label, fmt='{:.3f}', lat_fmt='{:.1f}'):
    cols = list(df.columns)
    best = {c: df.loc[[r for r in df.index if r != 'Oracle'], c].idxmax() for c in cols if c != 'Latency (ms)'}
    lines = ['\\begin{table}[t]', '\\centering', '\\caption{' + _esc(caption) + '}', '\\label{' + label + '}',
             '\\begin{tabular}{l' + 'c' * len(cols) + '}', '\\toprule',
             ' & ' + ' & '.join(_esc(c) for c in cols) + ' \\\\', '\\midrule']
    for r in df.index:
        cells = []
        for c in cols:
            v = df.loc[r, c]
            s = lat_fmt.format(v) if c == 'Latency (ms)' else fmt.format(v)
            if c != 'Latency (ms)' and best.get(c) == r:
                s = '\\textbf{' + s + '}'
            cells.append(s)
        lines.append(_esc(r) + ' & ' + ' & '.join(cells) + ' \\\\')
        if r == 'Rerank':
            lines.append('\\midrule')
    lines += ['\\bottomrule', '\\end{tabular}', '\\end{table}']
    return '\n'.join(lines)

def significance_table(RES):
    rows = []
    for m, dd in RES.items():
        for d, r in dd.items():
            b = r['boot_learned_vs_best_fixed']
            rows.append(dict(model=m, dataset=d, best_fixed=r['best_fixed'],
                             best_fixed_ndcg=r['table'][r['best_fixed']]['ndcg'],
                             learned_ndcg=r['table']['Learned router']['ndcg'], diff=b['diff'],
                             ci_lo=b['ci_lo'], ci_hi=b['ci_hi'], significant=bool(b['ci_lo'] > 0),
                             oracle_ndcg=r['table']['Oracle']['ndcg'], oracle_gap_closed=r['oracle_gap_closed'],
                             min_cost_to_match=r['min_cost_to_match_best_fixed']))
    return pd.DataFrame(rows)

def lodo_table(LODO):
    rows = []
    for m, dd in LODO.items():
        for d, r in dd.items():
            rows.append(dict(model=m, held_out=d, best_fixed=r['best_fixed'], best_fixed_ndcg=r['best_fixed_ndcg'],
                             router_ndcg=r['router_ndcg'], diff=r['boot']['diff'], ci_lo=r['boot']['ci_lo'],
                             ci_hi=r['boot']['ci_hi'], significant=bool(r['boot']['ci_lo'] > 0)))
    return pd.DataFrame(rows)

def plot_pareto(RES, path_prefix):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    models = list(RES.keys()); dsets = list(RES[models[0]].keys())
    fig, axes = plt.subplots(len(models), len(dsets), figsize=(3.3 * len(dsets), 2.7 * len(models)), squeeze=False)
    for i, m in enumerate(models):
        for j, d in enumerate(dsets):
            ax = axes[i][j]; r = RES[m][d]
            sw = r['learned_sweep']
            ax.plot([s['cost'] for s in sw], [s['ndcg'] for s in sw], '-o', ms=3, lw=1.2, label='Learned router')
            for name, pts in r['margin_curves'].items():
                ax.plot([p['cost'] for p in pts], [p['ndcg'] for p in pts], '--', lw=1, label=name)
            for name, mk in [('Dense', 's'), ('Hybrid', '^'), ('Rerank', 'D')]:
                ax.scatter(r['table'][name]['cost'], r['table'][name]['ndcg'], marker=mk, s=28, zorder=5, label=name)
            ax.scatter(r['table']['Oracle']['cost'], r['table']['Oracle']['ndcg'], marker='*', s=70, c='k', zorder=6, label='Oracle')
            ax.set_xscale('log')
            if i == 0: ax.set_title(d, fontsize=9)
            if j == 0: ax.set_ylabel(m + '\nnDCG@10', fontsize=8)
            if i == len(models) - 1: ax.set_xlabel('Latency (ms/query)', fontsize=8)
            ax.tick_params(labelsize=7); ax.grid(alpha=0.3)
    h, l = axes[0][0].get_legend_handles_labels()
    fig.legend(h, l, loc='lower center', ncol=4, fontsize=7, frameon=False)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    fig.savefig(path_prefix + '.pdf'); fig.savefig(path_prefix + '.png', dpi=300)
    plt.close(fig)

def to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items() if not str(k).startswith('_')}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o


## Config

In [3]:
import torch, random, gc
from sentence_transformers import CrossEncoder

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '(use a GPU runtime for FiQA / mpnet)')

DATASETS = ['scifact', 'nfcorpus', 'arguana', 'fiqa']
MODELS = {'MiniLM-L6': 'sentence-transformers/all-MiniLM-L6-v2',
          'mpnet-base': 'sentence-transformers/all-mpnet-base-v2',
          'bge-small': 'BAAI/bge-small-en-v1.5'}
QUERY_PREFIX = {'bge-small': 'Represent this sentence for searching relevant passages: '}
ALPHA = 0.75          # fixed for all models/datasets (same as Exp 2-6)
SHORTLIST = 20        # candidates passed to the cross-encoder
MAX_QUERIES = 1000    # cap per dataset (random, seed 42)
N_LAT = 50            # queries used for latency measurement
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/AI_Project/RAG Research/Exp7'
os.makedirs(OUT, exist_ok=True)
CONFIG = dict(datasets=DATASETS, models=MODELS, alpha=ALPHA, shortlist=SHORTLIST, max_queries=MAX_QUERIES,
              reranker='cross-encoder/ms-marco-MiniLM-L-6-v2', device=DEVICE,
              gpu=(torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'))
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=256, device=DEVICE)

device: cuda (use a GPU runtime for FiQA / mpnet)
Mounted at /content/drive


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## Load BEIR datasets and run the full pipeline (this is the long cell)

In [1]:
def load_beir(name, max_queries=1000, seed=42):
    from beir import util
    from beir.datasets.data_loader import GenericDataLoader
    url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{name}.zip"
    path = util.download_and_unzip(url, '/content/beir')
    corpus, queries, qrels = GenericDataLoader(data_folder=path).load(split='test')
    doc_ids = list(corpus.keys()); id2idx = {d: i for i, d in enumerate(doc_ids)}
    docs = [(corpus[d].get('title', '') + ' ' + corpus[d].get('text', '')).strip() for d in doc_ids]
    qids = sorted(q for q in queries if q in qrels and any(s > 0 and d in id2idx for d, s in qrels[q].items()))
    if len(qids) > max_queries:
        qids = sorted(random.Random(seed).sample(qids, max_queries))
    rels = [{id2idx[d]: s for d, s in qrels[q].items() if s > 0 and d in id2idx} for q in qids]
    return dict(docs=docs, qs=[queries[q] for q in qids], rels=rels, exclude=[id2idx.get(q) for q in qids])

RES = {m: {} for m in MODELS}
SETS = {m: {} for m in MODELS}
for ds in DATASETS:
    data = load_beir(ds, MAX_QUERIES)
    print(f"\n##### {ds}: {len(data['docs'])} docs, {len(data['qs'])} queries")
    bm = BM25Index(data['docs'])
    for mname, mpath in MODELS.items():
        print(f"--- {ds} / {mname}")
        pref = QUERY_PREFIX.get(mname, '')
        st = run_stack(mpath, data['docs'], data['qs'], bm, reranker, exclude_idx=data['exclude'],
                       alpha=ALPHA, shortlist=SHORTLIST, q_prefix=pref, device=DEVICE)
        ev = [eval_lists(st[k], data['rels']) for k in ('top_d', 'top_h', 'top_r')]
        ND = np.stack([e[0] for e in ev], 1); RC = np.stack([e[1] for e in ev], 1); MR = np.stack([e[2] for e in ev], 1)
        X = build_features(st['S_d'], st['S_b'], st['H'], data['qs'], bm)
        lat = measure_latency(st['model'], st['D'], bm, reranker, data['docs'], data['qs'][:N_LAT],
                              alpha=ALPHA, shortlist=SHORTLIST, q_prefix=pref)
        RES[mname][ds] = analyze_routing(ND, RC, MR, lat, X)
        SETS[mname][ds] = dict(ND=ND, X=X, t=lat)
        tb = RES[mname][ds]['table']
        print("   nDCG@10  dense %.3f | hybrid %.3f | rerank %.3f | learned %.3f | oracle %.3f | latency(ms) %s" % (
            tb['Dense']['ndcg'], tb['Hybrid']['ndcg'], tb['Rerank']['ndcg'], tb['Learned router']['ndcg'],
            tb['Oracle']['ndcg'], [round(x, 1) for x in lat]))
        del st; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()

NameError: name 'MODELS' is not defined

## Leave-one-dataset-out router transfer
Router trained on 3 datasets, tested on the 4th (features are label-free score/query statistics). This is the strongest test of whether routing generalizes beyond one domain.

In [5]:
LODO = {m: lodo_routing(SETS[m]) for m in MODELS}
print(lodo_table(LODO).round(3).to_string())

         model  held_out best_fixed  best_fixed_ndcg  router_ndcg   diff  ci_lo  ci_hi  significant
0    MiniLM-L6   scifact     Hybrid            0.714        0.704 -0.010 -0.026  0.005        False
1    MiniLM-L6  nfcorpus     Rerank            0.353        0.353 -0.000 -0.007  0.006        False
2    MiniLM-L6   arguana     Hybrid            0.534        0.534  0.000  0.000  0.000        False
3    MiniLM-L6      fiqa     Hybrid            0.393        0.396  0.003 -0.010  0.017        False
4   mpnet-base   scifact     Hybrid            0.724        0.713 -0.011 -0.022 -0.002        False
5   mpnet-base  nfcorpus     Hybrid            0.363        0.364  0.001 -0.010  0.012        False
6   mpnet-base   arguana     Hybrid            0.536        0.533 -0.003 -0.010  0.004        False
7   mpnet-base      fiqa      Dense            0.493        0.466 -0.026 -0.043 -0.010        False
8    bge-small   scifact     Hybrid            0.728        0.728  0.000 -0.018  0.016        False


## Tables, LaTeX, Pareto figure, download

In [6]:

os.makedirs(OUT, exist_ok=True)

for metric, mname_ in [('ndcg', 'nDCG@10'), ('recall', 'Recall@10'), ('mrr', 'MRR@10')]:
    for m in RES:
        df = main_table(RES[m], metric)
        df.to_csv(f'{OUT}/table_{metric}_{m}.csv')
        with open(f'{OUT}/table_{metric}_{m}.tex', 'w') as f:
            f.write(df_to_latex(df, f'{mname_} with {m} embeddings', f'tab:{metric}_{m}'))
        if metric == 'ndcg':
            print(f'\n===== {mname_} | {m} =====')
            print(df.round(3).to_string())

sig = significance_table(RES); sig.to_csv(f'{OUT}/significance_learned_vs_best_fixed.csv', index=False)
print('\n===== Learned router vs best fixed strategy (paired bootstrap, 95% CI) =====')
print(sig.round(3).to_string())

if LODO:
    lt = lodo_table(LODO); lt.to_csv(f'{OUT}/leave_one_out_router.csv', index=False)
    print('\n===== Leave-one-out router (trained on the other sets only) =====')
    print(lt.round(3).to_string())

plot_pareto(RES, f'{OUT}/pareto')
with open(f'{OUT}/all_results.json', 'w') as f:
    json.dump(to_jsonable({'results': RES, 'lodo': LODO, 'config': CONFIG}), f, indent=1)
print('\nAll outputs saved to Google Drive folder:', OUT)
print(sorted(os.listdir(OUT)))



===== nDCG@10 | MiniLM-L6 =====
                scifact  nfcorpus  arguana   fiqa    Avg  Latency (ms)
Dense             0.645     0.316    0.505  0.369  0.459        13.016
Hybrid            0.714     0.348    0.534  0.393  0.497        14.108
Rerank            0.691     0.353    0.516  0.383  0.486        98.542
Rule router       0.684     0.353    0.534  0.395  0.492        54.510
Learned router    0.711     0.358    0.529  0.392  0.498        44.582
Oracle            0.775     0.392    0.681  0.475  0.581        31.272

===== nDCG@10 | mpnet-base =====
                scifact  nfcorpus  arguana   fiqa    Avg  Latency (ms)
Dense             0.641     0.335    0.478  0.493  0.487        27.433
Hybrid            0.724     0.363    0.536  0.487  0.528        28.840
Rerank            0.692     0.362    0.512  0.414  0.495       129.253
Rule router       0.699     0.362    0.536  0.449  0.512        77.782
Learned router    0.714     0.365    0.528  0.489  0.524        56.478
Oracle    

## How to write this up (IEEE)
- Main result table = `table_ndcg_*.tex` (best non-oracle value per column is bold). Figure = `pareto.pdf` (vector, IEEE-friendly).
- Report **hardware + measured latency** (`CONFIG` in `all_results.json`), fixed alpha=0.75, single fixed reranker.
- Claim the router only where the bootstrap CI excludes 0; otherwise report "headroom exists, learned router captures X% of the oracle gap" (`oracle_gap_closed`).
- The leave-one-dataset-out table is your generalization evidence.